# Step 42: GREEN (4-bit) on a Colab T4Pre-registration: `results/partA/step42_green/PREREG.md` (committed before any score). **Every result is "GREEN (4-bit)", not the official fp16 GREEN.**How to run (the whole run is about 3 hours; scores are saved to Google Drive after every batch, so a disconnect loses at most one batch):1. Runtime > Change runtime type > **T4 GPU**.2. Put **`green_pairs.jsonl`** and **`dev_timing_pairs.jsonl`** in Google Drive in the folder `MyDrive/step42/` (nothing else is needed; no HiSBreast text is in them).3. Run the cells in order. After the install cell (cell 3) choose **Runtime > Restart session**, then continue from cell 4 (do not re-run the install).4. If the session disconnects, reconnect (T4 again), run cells 1, 4, 5, 6 (skip the install and the dev timing) and then cell 8: it resumes from the last saved id.5. When cell 8 finishes, download **`green_scores.jsonl`**, **`n_chosen.json`**, **`dev_timing.json`** and **`colab_env.json`** from `MyDrive/step42/` and put them in `usmeta/local_data/step42/`.

In [ ]:
# 1. Google Drive and paths
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/step42'
os.makedirs(WORK, exist_ok=True)
for f in ('green_pairs.jsonl', 'dev_timing_pairs.jsonl'):
    assert os.path.exists(f'{WORK}/{f}'), f'upload {f} to MyDrive/step42/ first'
print('files found in', WORK, os.listdir(WORK))

In [ ]:
# 2. GPU check (must say Tesla T4 or another CUDA GPU)
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)

In [ ]:
# 3. INSTALL (run once; then Runtime > Restart session and continue from cell 4)
!pip -q install bitsandbytes accelerate
!pip -q install "git+https://github.com/Stanford-AIMI/GREEN.git"

In [ ]:
# 4. imports, versions, environment record
import torch, json, time, os, re, sys, platform, importlib.metadata as im
WORK = '/content/drive/MyDrive/step42'
assert torch.cuda.is_available(), 'no GPU: Runtime > Change runtime type > T4 GPU'
env = dict(torch=torch.__version__, cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0), gpu_memory_gb=round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), python=sys.version.split()[0], platform=platform.platform())
for p in ('transformers', 'bitsandbytes', 'accelerate', 'green_score', 'numpy', 'datasets'):
    try: env[p] = im.version(p)
    except Exception as e: env[p] = f'not installed ({e})'
json.dump(env, open(f'{WORK}/colab_env.json', 'w'), indent=1); print(json.dumps(env, indent=1))

In [ ]:
# 5. load GREEN-radllama2-7b in 4-bit (NF4) and wire it into the GREEN class (only the loading differs from the package)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from green_score import GREEN
MODEL = 'StanfordAIMI/GREEN-radllama2-7b'
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
g = GREEN(model_name=None, output_dir='/content/green_out', compute_summary_stats=False)
g.model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map={'': 0}, torch_dtype=torch.float16, trust_remote_code=True)
g.model.eval()
tok = AutoTokenizer.from_pretrained(MODEL, add_eos_token=True, use_fast=True, trust_remote_code=True, padding_side='left')
tok.chat_template = "{% for message in messages %}\n{% if message['from'] == 'human' %}\n{{ '<|user|>\n' + message['value'] + eos_token }}\n{% elif message['from'] == 'system' %}\n{{ '<|system|>\n' + message['value'] + eos_token }}\n{% elif message['from'] == 'gpt' %}\n{{ '<|assistant|>\n'  + message['value'] + eos_token }}\n{% endif %}\n{% if loop.last and add_generation_prompt %}\n{{ '<|assistant|>' }}\n{% endif %}\n{% endfor %}"
tok.pad_token = tok.eos_token
tok.clean_up_tokenization_spaces = True
g.tokenizer = tok
g.model_name = 'GREEN-radllama2-7b'
g.batch_size = 8
print('GREEN (4-bit) loaded; GPU memory used (GB):', round(torch.cuda.memory_allocated() / 1e9, 2))

In [ ]:
# 6. scoring helpers: read pairs, score a batch (falls back to batch size 4 if the GPU runs out of memory), append-only save
def read_jsonl(p): return [json.loads(l) for l in open(p) if l.strip()]

def score_rows(rows):
    refs = [r['reference'] for r in rows]; hyps = [r['candidate'] for r in rows]
    mean, std, scores, summary, df = g(refs, hyps)
    out = []
    for r, s, (_, row) in zip(rows, scores, df.iterrows()):
        out.append(dict(id=r['id'], green_score=s, counts={c: int(row[c]) for c in g.sub_categories + ['Matched Findings']}, analysis=row['green_analysis']))
    return out

def score_safe(rows):
    try: return score_rows(rows)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); g.batch_size = 4; print('out of memory: batch size 4')
        return score_rows(rows)

def append_scores(path, out):
    with open(path, 'a') as fh:
        for o in out: fh.write(json.dumps(o) + '\n')
        fh.flush(); os.fsync(fh.fileno())

In [ ]:
# 7. dev timing on 20 templated DEV pairs (not the test pairs) and the pre-registered size rule
dev = read_jsonl(f'{WORK}/dev_timing_pairs.jsonl'); t0 = time.time(); dev_out = []
for i in range(0, len(dev), 8):
    b = time.time(); dev_out += score_safe(dev[i:i + 8]); print('dev batch', i // 8 + 1, round(time.time() - b, 1), 's')
sec = time.time() - t0; s_pair = sec / len(dev)
ident = [o['green_score'] for o, r in zip(dev_out, dev) if r['reference'] == r['candidate']]
json.dump(dict(pairs=len(dev), seconds=sec, seconds_per_pair=s_pair, batch_size=g.batch_size, identical_copy_scores=ident, unparsed=sum(o['green_score'] is None for o in dev_out)), open(f'{WORK}/dev_timing.json', 'w'), indent=1)
with open(f'{WORK}/dev_green_scores.jsonl', 'w') as fh:
    for o in dev_out: fh.write(json.dumps(o) + '\n')
# pre-registered rule: s = 1.1 x seconds per pair; n_units = largest prefix of the unit order with pairs x s <= 10,000 s; at least 40 units; at most all
pairs = read_jsonl(f'{WORK}/green_pairs.jsonl'); units = []; seen = set()
for p in pairs:
    u = p['id'].split('_')[0]
    if u not in seen: seen.add(u); units.append(u)
size = {u: 0 for u in units}
for p in pairs: size[p['id'].split('_')[0]] += 1
s_eff = 1.1 * s_pair; cum = 0; n_units = 0
for u in units:
    if (cum + size[u]) * s_eff > 10000: break
    cum += size[u]; n_units += 1
n_units = max(40, n_units) if len(units) >= 40 else len(units)
chosen = units[:n_units]; n_pairs = sum(size[u] for u in chosen)
nc = dict(rule='n_units = largest prefix with pairs x 1.1 x s <= 10000 s, >= 40 units', seconds_per_pair_dev=s_pair, s_eff=s_eff, n_units=n_units, n_pairs=n_pairs, expected_hours=n_pairs * s_eff / 3600, units=chosen, written_at=time.strftime('%Y-%m-%d %H:%M:%S'))
if not os.path.exists(f'{WORK}/n_chosen.json'): json.dump(nc, open(f'{WORK}/n_chosen.json', 'w'), indent=1)   # written once, BEFORE any test pair is scored
print(json.dumps({k: v for k, v in json.load(open(f'{WORK}/n_chosen.json')).items() if k != 'units'}, indent=1)); print('dev identical-copy GREEN scores (about 1.0 expected):', ident)

In [ ]:
# 8. score the chosen test pairs; resumable (saves after every batch); hard stop at 11,000 s of scoring in this session
nc = json.load(open(f'{WORK}/n_chosen.json')); chosen = set(nc['units']); pairs = [p for p in read_jsonl(f'{WORK}/green_pairs.jsonl') if p['id'].split('_')[0] in chosen]
OUTP = f'{WORK}/green_scores.jsonl'; done = {json.loads(l)['id'] for l in open(OUTP)} if os.path.exists(OUTP) else set()
todo = [p for p in pairs if p['id'] not in done]; print(len(pairs), 'pairs chosen;', len(done), 'already scored;', len(todo), 'to do')
t0 = time.time(); HARD = 11000
for i in range(0, len(todo), g.batch_size):
    if time.time() - t0 > HARD: print('hard stop reached: rerun this cell (after a reconnect) to resume'); break
    batch = todo[i:i + g.batch_size]; out = score_safe(batch); append_scores(OUTP, out)
    if (i // g.batch_size) % 10 == 0: print(f'{len(done) + i + len(batch)} / {len(pairs)} scored, {time.time() - t0:.0f} s this session, {sum(o["green_score"] is None for o in out)} unparsed in the last batch', flush=True)
print('finished this session:', sum(1 for _ in open(OUTP)), 'of', len(pairs), 'pairs scored')

In [ ]:
# 9. summary (then download green_scores.jsonl, n_chosen.json, dev_timing.json, colab_env.json from MyDrive/step42/)
rows = [json.loads(l) for l in open(OUTP)]
print(len(rows), 'scored;', sum(r['green_score'] is None for r in rows), 'unparsed;', sum(r['green_score'] == 0 for r in rows), 'scored 0;', 'mean GREEN (4-bit) score', sum(r['green_score'] for r in rows if r['green_score'] is not None) / max(1, sum(r['green_score'] is not None for r in rows)))